# 05 EASE, ItemKNN, and Metadata Paths

This notebook documents the remaining team variants and separates the current retained models from earlier metadata experiments.

## Covered paths
- **Plain EASE** from `phase2/run_ease.py`
- **EASE with metadata** from `phase2/run_ease_metadata.py` and `phase2/run_ease_metadata_best.py`
- **ItemKNN BM25** from `phase2/run_itemknn.py`
- **old LightFM hybrid** from `variants/variant_c_hybrid/train_hybrid.py`

The repo contains both old metadata work and newer phase2 metadata runners. This notebook keeps those paths distinct so a reader can understand what was retained, what was exploratory, and how each model family fits into the final comparison.

In [1]:
import json

import pandas as pd
from IPython.display import display, Markdown

from analysis.shared_utils import PROJECT_ROOT, RESULTS_DIR, historical_variant_c_results, load_retained_results

retained_test = load_retained_results("test")
ease_row = retained_test.loc[retained_test["model"] == "EASE-Binary-3000"]
itemknn_row = retained_test.loc[retained_test["model"] == "ItemKNN-BM25-K320"]
ease_meta_sweep_row = pd.read_csv(RESULTS_DIR / "ease_metadata_test_best.csv")
ease_meta_prior_row = pd.read_csv(RESULTS_DIR / "ease_metadata_best_test.csv")
variant_c_history = historical_variant_c_results()

results_dir = PROJECT_ROOT / "variants" / "variant_c_hybrid"
with open(results_dir / "results_itemknn_selected.json", "r", encoding="utf-8") as f:
    itemknn_json = json.load(f)
with open(results_dir / "results_hybrid.json", "r", encoding="utf-8") as f:
    hybrid_json = json.load(f)

current_variant_rows = pd.concat(
    [ease_row, ease_meta_sweep_row, ease_meta_prior_row, itemknn_row],
    ignore_index=True,
    sort=False,
)

display(Markdown("## EASE, EASE with metadata, and ItemKNN on the test split"))
display(current_variant_rows)

display(Markdown("## old Variant C story"))
display(variant_c_history)

print("Current `results_itemknn_selected.json` variant:", itemknn_json.get("variant"))
print("Current `results_hybrid.json` variant:", hybrid_json.get("variant"))

## EASE, EASE with metadata, and ItemKNN on the test split

,model,precision@10,recall@10,ndcg@10,users_evaluated,eval_split,metadata_mode,fusion,alpha
0,EASE-Binary-3000,0.044746,0.065984,0.086413,7080,test,NaN,NaN,NaN
1,EASEMeta-category_char-score_blend-a0.5,0.044831,0.065467,0.086227,7080,test,category_char,score_blend,0.5
2,EASEMeta-ratingPrior-a0.03,0.044929,0.066133,0.086694,7080,test,NaN,NaN,NaN
3,ItemKNN-BM25-K320,0.042994,0.063505,0.083244,7080,test,NaN,NaN,NaN


## old Variant C story

,variant_story,precision@10,recall@10,ndcg@10
0,LightFM hybrid with category metadata,0.036384,0.051236,0.052665
1,Retained ItemKNN BM25,0.042994,0.063505,0.065180


Current `results_itemknn_selected.json` variant: itemknn_bm25_selected
Current `results_hybrid.json` variant: itemknn_bm25_selected


In [2]:
teammate_summary = pd.DataFrame(
    [
        {
            "model": "EASE-Binary-3000",
            "core_idea": "closed-form linear item-item recommender from the train interaction matrix",
            "strength": "strong plain interaction-only anchor and top retained baseline among the original three variants",
            "risk": "still head-item heavy without any side-information help",
        },
        {
            "model": "EASEMeta-ratingPrior-a0.03",
            "core_idea": "base EASE scores plus a small metadata-derived item prior",
            "strength": "tiny positive gain over plain EASE while satisfying the metadata requirement",
            "risk": "gain is small, so the explanation has to focus on design intent as well as numbers",
        },
        {
            "model": "ItemKNN-BM25-K320",
            "core_idea": "item-item nearest neighbors over a BM25-weighted item-user matrix",
            "strength": "simple and interpretable, beats UserCF slightly",
            "risk": "ranking quality trails the strongest EASE / RP3beta paths",
        },
        {
            "model": "LightFM hybrid (old)",
            "core_idea": "item identity embeddings plus category metadata features",
            "strength": "shows the team explored metadata explicitly",
            "risk": "underperformed and the checked-in JSON naming is inconsistent",
        },
    ]
)

display(teammate_summary)



,model,core_idea,strength,risk
0,EASE-Binary-3000,closed-form linear item-item recommender from ...,strong plain interaction-only anchor and top r...,still head-item heavy without any side-informa...
1,EASEMeta-ratingPrior-a0.03,base EASE scores plus a small metadata-derived...,tiny positive gain over plain EASE while satis...,"gain is small, so the explanation has to focus..."
2,ItemKNN-BM25-K320,item-item nearest neighbors over a BM25-weight...,"simple and interpretable, beats UserCF slightly",ranking quality trails the strongest EASE / RP...
3,LightFM hybrid (old),item identity embeddings plus category metadat...,shows the team explored metadata explicitly,underperformed and the checked-in JSON naming ...


## Interpretation Notes

### Plain EASE
EASE is the strongest clean interaction-only item-item model in the repo. It learns a regularized linear item-item weight matrix from the train interaction matrix and performs especially well on this sparse implicit dataset.

### EASE with metadata
The pushed repository adds two metadata-aware EASE paths: a metadata-similarity fusion/reranking sweep and a compact rating-prior augmentation. The rating-prior variant slightly improves over plain EASE, showing that metadata can add value when used as a small auxiliary signal.

### ItemKNN BM25
ItemKNN BM25 uses BM25-weighted item-user interactions before computing item-item nearest neighbors. It remains simple and interpretable, slightly improves over `UserCF`, and provides a local-neighborhood contrast to EASE and RP3beta.

### Historical Variant C metadata path
Variant C originally included a LightFM hybrid with category metadata. That path is still documented in `variants/variant_c_hybrid/`, but the final retained Variant C model is ItemKNN BM25 because it produced stronger ranking results under the shared protocol.

### Repo interpretation
When reading this repo, treat metadata as appearing in two places: the historical LightFM exploration under `variants/`, and the newer phase2 metadata-aware EASE/RP3beta runners.

## Model Intuition Summary

### EASE
EASE solves for a regularized item-item weight matrix in closed form. Each candidate app is scored from the apps already present in the user's history. This gives a global linear item-item reconstruction model.

### EASE with metadata
The metadata-aware EASE variants keep the same linear core and add side information after the base score is computed. The best pushed path adds a small metadata-derived rating prior, which slightly improves the shared ranking metrics.

### ItemKNN BM25
ItemKNN first reweights the item-user matrix with BM25, then computes item-item nearest neighbors. Candidate apps are scored by aggregating similarity from the user's seen apps. This is local and interpretable, unlike EASE's global linear reconstruction.

### Contrast across methods
- `EASE`: global linear item-item reconstruction.
- `EASE + metadata`: EASE plus a small side-information signal.
- `ItemKNN`: local item-neighborhood voting.
- `RP3beta`: graph-based random walk with popularity control.

In [3]:
ease_compare = pd.concat([ease_row, ease_meta_sweep_row, ease_meta_prior_row], ignore_index=True, sort=False)
base_ease_ndcg = float(ease_compare.loc[ease_compare["model"] == "EASE-Binary-3000", "ndcg@10"].iloc[0])
base_ease_precision = float(ease_compare.loc[ease_compare["model"] == "EASE-Binary-3000", "precision@10"].iloc[0])
base_ease_recall = float(ease_compare.loc[ease_compare["model"] == "EASE-Binary-3000", "recall@10"].iloc[0])

ease_compare["ndcg_delta_vs_plain_ease"] = ease_compare["ndcg@10"] - base_ease_ndcg
ease_compare["precision_delta_vs_plain_ease"] = ease_compare["precision@10"] - base_ease_precision
ease_compare["recall_delta_vs_plain_ease"] = ease_compare["recall@10"] - base_ease_recall

print("Plain EASE versus pushed metadata-aware EASE variants")
display(ease_compare.sort_values("ndcg@10", ascending=False))

Plain EASE versus pushed metadata-aware EASE variants


,model,precision@10,recall@10,ndcg@10,users_evaluated,eval_split,metadata_mode,fusion,alpha,ndcg_delta_vs_plain_ease,precision_delta_vs_plain_ease,recall_delta_vs_plain_ease
2,EASEMeta-ratingPrior-a0.03,0.044929,0.066133,0.086694,7080,test,NaN,NaN,NaN,0.000282,0.000184,0.000149
0,EASE-Binary-3000,0.044746,0.065984,0.086413,7080,test,NaN,NaN,NaN,0.000000,0.000000,0.000000
1,EASEMeta-category_char-score_blend-a0.5,0.044831,0.065467,0.086227,7080,test,category_char,score_blend,0.5,-0.000186,0.000085,-0.000517
